# Fraud Detection — LightGBM Training on Feast Offline Features

Trains a LightGBM binary classifier on **point-in-time features pulled from the Feast
offline store** (`get_historical_features`), joined to ground-truth labels in
`application.labels`. Artifacts are written to `models/`.

**Pipeline**
1. **Build the entity spine** — one row per *labeled* transaction, carrying the Feast
   entity join keys (`user_id`, `card_id`), the transaction `event_timestamp`
   (`created_at`), and the `label`.
2. **Feast offline retrieval** — `store.get_historical_features(spine, service)` runs a
   **point-in-time join**: for every transaction it returns the feature values that were
   valid *as of that transaction's own `created_at`*. Because the source's event field is
   `created_at` (per-transaction) — not `computed_at` (the ETL/materialize batch time) —
   each row gets its own velocity/graph features rather than a single collapsed vector.
3. **Temporal split** — train on the earlier rows, validate on the most recent, mirroring
   production where the model always scores the future.
4. **Train** a LightGBM classifier (binary — the task is fraud vs. legit).
5. **Evaluate** on the held-out validation slice (ROC-AUC, PR-AUC, precision@k).
6. **Save** the booster to `models/model.bst` (+ metrics & feature schema) and **log the
   run to MLflow** (params, metrics, model, artifacts).

**Design notes**
- Raw entity IDs (`transaction_id`, `user_id`, `card_id`, ...) are used only to build the
  spine and are dropped from the feature matrix — entity behaviour is already encoded by
  the velocity features, and raw IDs invite identity overfitting.
- `is_declined` is not part of the Feast feature view (it is the current transaction's
  post-authorization status — leakage for a pre-auth scorer). The leakage-safe historical
  `card_declines_24h` is kept.


In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd
import psycopg
from feast import FeatureStore
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
CURRENT_DIR = Path().cwd()
# When run from scripts/training/, hop up to the repo root; else assume cwd is the root.
REPO_ROOT = CURRENT_DIR.parents[1] if CURRENT_DIR.name == "training" else CURRENT_DIR
MODEL_DIR = REPO_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FEAST_REPO = REPO_ROOT / "src" / "feature_store"


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(REPO_ROOT / ".env")

CONNINFO = (
    f"host={os.getenv('POSTGRES_HOST')} port={os.getenv('POSTGRES_PORT')} "
    f"user={os.getenv('POSTGRES_USER')} password={os.getenv('POSTGRES_PASSWORD')} "
    f"dbname={os.getenv('POSTGRES_DB')}"
)


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# MLflow — tracking server runs on a VM, port-forwarded to localhost:5000.
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "fraud-detection")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

log(f"Feast repo    -> {FEAST_REPO}")
log(f"Model output  -> {MODEL_DIR}")
log(f"MLflow server -> {MLFLOW_TRACKING_URI} (experiment: {MLFLOW_EXPERIMENT_NAME})")


In [ ]:
SEED = 42
TARGET = "label"
FEATURE_SERVICE = "fraud_detection_service"
# Name under which each finished run is registered in the MLflow Model Registry.
MODEL_NAME = os.getenv("MODEL_NAME", "fraud-detection-lightgbm")

# Identity / bookkeeping columns returned alongside the features — never model inputs.
# (The direct-Postgres fallback SELECT f.* also returns merchant_id, device_id, is_declined
#  and computed_at, which the Feast feature view omits; excluding them here keeps both
#  loading paths on an identical feature set. is_declined is post-auth leakage.)
NON_FEATURE_COLS = {
    "transaction_id", "user_id", "card_id", "merchant_id", "device_id",
    "event_timestamp", "computed_at", "is_declined", TARGET,
}

# Low-cardinality text/boolean features -> label-encoded, passed as LightGBM categoricals.
CATEGORICAL_COLS = [
    "channel", "card_brand", "card_type",
    "customer_segment", "merchant_category",
    "is_virtual", "email_verified",
]

LGB_PARAMS = {
    "objective": "binary",     # fraud vs. legit -> binary classification
    # Track both, but early-stop on average_precision (PR-AUC): ROC-AUC saturates within a
    # handful of trees under the heavy scale_pos_weight and would stop training far too
    # early, starving the minority-class ranking that PR-AUC measures.
    "metric": ["average_precision", "auc"],
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 128,
    "min_child_samples": 100,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
    "n_jobs": -1,
    "seed": SEED,
}
NUM_BOOST_ROUND = 5000
EARLY_STOPPING = 200
VAL_FRAC = 0.20   # most-recent slice held out for early stopping + evaluation


## 1 — Build the entity spine (labeled transactions)

The spine is the *entity dataframe* Feast joins features onto: one row per labeled
transaction with the entity join keys, the point-in-time `event_timestamp`
(`created_at`), and the ground-truth `label`. We read it from `transaction_features`
(for the entity keys + `created_at`) joined to `application.labels`, so every spine row is
guaranteed to have a matching feature row to retrieve.

It is bounded to a short, fully-labeled **time window** (`SPINE_START` .. `SPINE_END`):
Feast's Postgres point-in-time join gets expensive over the full history, so a recent
window keeps retrieval fast while staying representative.

In [ ]:
# Training window: a recent, fully-labeled slice. Feast's Postgres point-in-time join is
# expensive over the whole history (cost grows with the square of transactions per
# (user, card) pair), so we bound the spine to a short window — retrieval stays fast and
# the slice is still representative. We stop before the label horizon: fraud labels arrive
# post-hoc (chargebacks / review), so the most recent transactions are not yet labeled.
SPINE_START = "2026-06-27 00:00:00+00"
SPINE_END = "2026-06-29 00:00:00+00"

SPINE_SQL = """
    SELECT f.transaction_id,
           f.user_id,
           f.card_id,
           f.created_at AS event_timestamp,
           l.label
    FROM application.transaction_features f
    JOIN application.labels l ON l.transaction_id = f.transaction_id
    WHERE l.label IS NOT NULL
      AND f.created_at >= %(start)s
      AND f.created_at <  %(end)s
    ORDER BY f.created_at
"""


def load_spine() -> pd.DataFrame:
    log(f"Building entity spine (window {SPINE_START[:10]} .. {SPINE_END[:10]}) ...")
    with psycopg.connect(CONNINFO) as conn:
        spine = pd.read_sql(SPINE_SQL, conn, params={"start": SPINE_START, "end": SPINE_END})
    spine["event_timestamp"] = pd.to_datetime(spine["event_timestamp"], utc=True)
    spine[TARGET] = spine[TARGET].astype(np.int8)
    pos = int(spine[TARGET].sum())
    log(f"  spine rows: {len(spine):,}  positives: {pos:,} ({pos / len(spine):.3%})")
    return spine.reset_index(drop=True)


spine = load_spine()
spine.head()


## 2 — Retrieve point-in-time features from the Feast offline store

`get_historical_features` uploads the entity keys + `event_timestamp` and runs a
point-in-time join against the Postgres offline store, returning each transaction's
features **as of its own `created_at`**. We then **merge the ground-truth labels** back on
by `transaction_id` to form the labeled training frame.

> ⚠️ **Temporary:** the Feast retrieval is commented out below — the Postgres offline
> point-in-time join is too slow over the full history. As a stopgap we load all labeled
> features **directly from Postgres** (same columns, whole history). Re-enable the Feast
> block once retrieval performance is addressed.

In [ ]:
def load_training_frame(spine: pd.DataFrame) -> pd.DataFrame:
    # --- Feast offline retrieval (TEMPORARILY DISABLED) --------------------------------
    # The Feast Postgres point-in-time join is too slow over the full history (its cost
    # grows with the square of transactions per (user, card) pair). Until that is
    # addressed, we bypass Feast and read ALL labeled features straight from Postgres.
    # Re-enable the block below (and drop the fallback) to restore the Feast path.
    #
    # store = FeatureStore(repo_path=str(FEAST_REPO))
    # service = store.get_feature_service(FEATURE_SERVICE)
    # log(f"Feast point-in-time join via feature service '{FEATURE_SERVICE}' ...")
    # entity_df = spine[["transaction_id", "user_id", "card_id", "event_timestamp"]]
    # features = store.get_historical_features(entity_df=entity_df, features=service).to_df()
    # df = features.merge(spine[["transaction_id", TARGET]], on="transaction_id", how="inner")
    # ----------------------------------------------------------------------------------

    # TEMP fallback: load all labeled features directly from Postgres (whole history).
    fallback_sql = """
        SELECT f.*, l.label
        FROM application.transaction_features f
        JOIN application.labels l ON l.transaction_id = f.transaction_id
        WHERE l.label IS NOT NULL
        ORDER BY f.created_at
    """
    log("TEMP: loading ALL labeled features directly from Postgres (Feast bypass) ...")
    with psycopg.connect(CONNINFO) as conn:
        df = pd.read_sql(fallback_sql, conn)
    df = df.rename(columns={"created_at": "event_timestamp"})

    df[TARGET] = df[TARGET].astype(np.int8)
    df = (
        df.drop_duplicates(subset="transaction_id")
          .sort_values("event_timestamp")
          .reset_index(drop=True)
    )
    log(f"  loaded {len(df):,} rows x {df.shape[1]} cols from Postgres")
    return df


df = load_training_frame(spine)
df.head()


## 3 — Encode categoricals and select feature columns

In [ ]:
def encode_categoricals(df: pd.DataFrame) -> tuple[list[str], dict[str, dict[str, int]]]:
    """Label-encode the declared categorical columns; return (cat_cols, encoders).

    Booleans and text are stringified first so NULL/None maps to an explicit 'missing'
    bucket and encodings are stable across runs.
    """
    encoders: dict[str, dict[str, int]] = {}
    cat_cols: list[str] = []
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            continue
        s = df[col].astype("string").fillna("missing").astype(str)
        mapping = {v: i for i, v in enumerate(sorted(s.unique()))}
        df[col] = s.map(mapping).astype(np.int32)
        encoders[col] = mapping
        cat_cols.append(col)
    log(f"Label-encoded {len(cat_cols)} categorical columns.")
    return cat_cols, encoders


def feature_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in NON_FEATURE_COLS]


cat_cols, encoders = encode_categoricals(df)
feat_cols = feature_columns(df)
log(f"{len(feat_cols)} feature columns ({len(cat_cols)} categorical)")
feat_cols


## 4 — Temporal train / validation split

A chronological split (train on the earliest `1 - VAL_FRAC`, validate on the most recent
`VAL_FRAC`) mirrors production, where the model only ever scores transactions that occur
after the ones it was trained on. A random split would leak future behaviour.

In [ ]:
def temporal_split(df: pd.DataFrame, val_frac: float = VAL_FRAC):
    """Chronological split: earliest (1-val_frac) -> train, most-recent val_frac -> val."""
    n = len(df)
    cut = int(n * (1.0 - val_frac))
    idx = np.arange(n)  # df is already sorted by event_timestamp
    tr, va = idx[:cut], idx[cut:]
    log(f"Split — train: {len(tr):,}  val: {len(va):,} (most-recent {val_frac:.0%})")
    return tr, va


def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k_frac: float) -> float:
    k = max(1, int(len(y_score) * k_frac))
    top = np.argpartition(-y_score, k - 1)[:k]
    return float(y_true[top].sum()) / k


train_idx, val_idx = temporal_split(df)


## 5 — Train on the training set, early-stop on validation

Fraud is heavily imbalanced (~0.5% positives), so `scale_pos_weight` is set from the
training split's class ratio. Boosting rounds are chosen by early stopping on the
validation slice.

In [ ]:
X, y = df[feat_cols], df[TARGET].to_numpy()
cat_idx = [c for c in cat_cols if c in feat_cols]

pos = int(y[train_idx].sum())
scale_pos_weight = (len(train_idx) - pos) / max(1, pos)
params = {**LGB_PARAMS, "scale_pos_weight": scale_pos_weight}
log(f"scale_pos_weight = {scale_pos_weight:.1f}  (train positives: {pos:,})")

# Open an MLflow run for this training job (end any run left open by a prior partial run,
# so re-executing this cell is idempotent). The run stays active across the eval + save
# cells below, which log metrics and the model into it.
mlflow.end_run()
run = mlflow.start_run(run_name=f"lgbm-{time.strftime('%Y%m%d-%H%M%S')}")
mlflow.set_tags({
    "model_type": "lightgbm",
    "task": "binary_fraud_classification",
    "data_source": "postgres_direct (feast bypass)",
})
mlflow.log_params({
    **params,
    "num_boost_round": NUM_BOOST_ROUND,
    "early_stopping_rounds": EARLY_STOPPING,
    "val_frac": VAL_FRAC,
    "n_features": len(feat_cols),
    "n_train": int(len(train_idx)),
    "n_val": int(len(val_idx)),
})
log(f"MLflow run started: {run.info.run_id}")

dtrain = lgb.Dataset(X.iloc[train_idx], label=y[train_idx], categorical_feature=cat_idx)
dvalid = lgb.Dataset(X.iloc[val_idx], label=y[val_idx], categorical_feature=cat_idx,
                     reference=dtrain)

booster = lgb.train(
    params,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[dtrain, dvalid],
    valid_names=["train", "valid"],
    callbacks=[
        # first_metric_only -> stop on average_precision (the first metric), not ROC-AUC.
        lgb.early_stopping(EARLY_STOPPING, first_metric_only=True),
        lgb.log_evaluation(period=100),
    ],
)
log(f"Best iteration: {booster.best_iteration}")


## 6 — Evaluate on the held-out validation set

In [ ]:
val_pred = booster.predict(X.iloc[val_idx], num_iteration=booster.best_iteration)
y_val = y[val_idx]

metrics = {
    "val_roc_auc": float(roc_auc_score(y_val, val_pred)),
    "val_pr_auc": float(average_precision_score(y_val, val_pred)),
    "val_precision_at_1pct": precision_at_k(y_val, val_pred, 0.01),
    "val_precision_at_0.5pct": precision_at_k(y_val, val_pred, 0.005),
    "best_iteration": int(booster.best_iteration),
    "n_features": len(feat_cols),
    "n_train": int(len(train_idx)),
    "n_val": int(len(val_idx)),
    "val_positives": int(y_val.sum()),
    "scale_pos_weight": float(scale_pos_weight),
}
mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
print(json.dumps(metrics, indent=2))


## 7 — Save the model, its schema, and log to MLflow

The evaluated booster is written to `models/model.bst` (LightGBM's native format),
alongside the validation metrics and a feature schema (column order + categorical
encoders) so the serving path can rebuild an identical feature vector. The same booster,
params, metrics and artifacts are also logged to the **MLflow** tracking server, and the
run is closed. The model is also **registered** in the MLflow Model Registry under
`MODEL_NAME`, creating a new version on every training run.

In [ ]:
booster.save_model(str(MODEL_DIR / "model.bst"), num_iteration=booster.best_iteration)

schema = {
    "version": 1,
    "offline_source": "feast::fraud_detection_service",
    "feature_view": "transaction_features",
    "label_table": "application.labels",
    "target": TARGET,
    "feature_columns": feat_cols,
    "categorical_features": cat_cols,
    "categorical_encoders": encoders,
    "lgb_params": LGB_PARAMS,
}

(MODEL_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(MODEL_DIR / "feature_schema.json").write_text(json.dumps(schema, indent=2))
log(f"Saved model.bst + metrics.json + feature_schema.json to {MODEL_DIR}")

# Log the model + artifacts to MLflow, register a new version, and close the run.
from mlflow.models import infer_signature

sample = X.iloc[val_idx].head(200)
signature = infer_signature(sample, booster.predict(sample))

model_info = mlflow.lightgbm.log_model(
    booster,
    artifact_path="model",
    signature=signature,
    input_example=X.iloc[val_idx].head(5),
    registered_model_name=MODEL_NAME,   # -> new registry version every run
)
for name in ("model.bst", "metrics.json", "feature_schema.json"):
    mlflow.log_artifact(str(MODEL_DIR / name))

run_id = mlflow.active_run().info.run_id
mlflow.end_run()
log(f"Logged + registered model '{MODEL_NAME}' from run {run_id} ({MLFLOW_TRACKING_URI})")
if getattr(model_info, "registered_model_version", None):
    log(f"Registered version: {MODEL_NAME} v{model_info.registered_model_version}")
